In [37]:
import pandas as pd 

In [38]:
pd.set_option('display.max_columns', None)

## Before starting this phase, make sure to run the following three scripts in sequence. They work with the data obtained from Kaggle:

### 1. First Script: Data Preprocessing (gen_data.py)
This script processes the dataset by filtering the data based on a list of target artists from Europe and Japan. It then assigns regions (Europe or Japan) to the corresponding artists and saves the cleaned data into a new folder.

#### Key steps:
- Filters the dataset to only include tracks by specific European and Japanese artists.
- Assigns each track to a region (Europe or Japan) based on the artist.
- Saves the filtered data into separate CSV files.

---

### 2. Second Script: Data Alignment (merge_data.py)
This script combines multiple filtered datasets into one comprehensive dataset, aligning the columns with the predefined features. It ensures that all the data from the different files are merged properly, with consistent column names.

#### Key steps:
- Reads multiple CSV files from the filtered data directory.
- Aligns the columns of each dataset to a predefined set of features.
- Combines all the aligned datasets into a single file and saves it.

---

### 3. Third Script: Fetching Missing Data from Spotify API (filled_data.py)
This script fetches additional track and artist data from Spotify using the Spotify API. Be aware that the API has request limits, so it’s better to process the data in smaller batches.

#### Key steps:
- Uses the Spotify API to fill in missing data, such as genre, popularity, explicit status, and release date.
- Makes requests to the API for each track, handling potential errors gracefully.
- Saves the updated dataset with the additional information fetched from Spotify.

---

### Note:
If the data has already been preprocessed and cleaned, you can skip the data cleaning step and directly use the prepared dataset from the previous phases.

# **Prepeared data**

In [39]:
df = pd.read_csv('../data/combined_filled_data.csv')
df.head()

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
0,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,70LcF31zb1H0PyJoS1Sx1r,Creep,1993-02-22,art rock,85.0,1.0,238640,0.0097,0.515,0.430,0.000133,7,0.1290,-9.935,1,0.0372,91.844,0.104
1,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,10nyNJ6zNy2YVYLrcwLccB,No Surprises,1997-05-28,art rock,82.0,0.0,229120,0.0577,0.255,0.393,0.003610,5,0.1130,-10.654,1,0.0278,76.426,0.118
2,0XNa1vTidXlvJ2gHSsRi4A,Franz Ferdinand,Europe,46gSk82duJtX3TTA182ruG,This fffire - New Version,2004-11-15,indie rock,76.0,0.0,218080,0.0259,0.442,0.887,0.000005,4,0.0655,-5.943,0,0.1000,146.358,0.637
3,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,63OQupATfueTdZMWTxW03A,Karma Police,1997-05-28,art rock,76.0,0.0,264066,0.0638,0.360,0.501,0.000093,7,0.1720,-9.129,1,0.0258,74.807,0.324
4,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,2a1iMaoWQ5MnvLFBDv4qkf,High and Dry,1995-03-13,art rock,75.0,0.0,257480,0.0724,0.419,0.383,0.017600,4,0.0896,-11.782,1,0.0256,87.568,0.350


## **Check dataset**

In [40]:
df.dtypes

Artist ID            object
Artist name          object
Region               object
Track ID             object
Track name           object
Release date         object
Genre                object
Popularity          float64
Explicit             object
Duration              int64
Acousticness        float64
Danceability        float64
Energy              float64
Instrumentalness    float64
Key                  object
Liveness            float64
Loudness            float64
Mode                 object
Speechiness         float64
Tempo               float64
Valence             float64
dtype: object

In [41]:
df.isna().sum()

Artist ID            0
Artist name          0
Region               0
Track ID             0
Track name           0
Release date         0
Genre               53
Popularity           0
Explicit             0
Duration             0
Acousticness         0
Danceability         0
Energy               0
Instrumentalness     0
Key                  0
Liveness             0
Loudness             0
Mode                 0
Speechiness          0
Tempo                0
Valence              0
dtype: int64

In [42]:
df['Mode'].value_counts()

Mode
1        939
Major    783
0        526
Minor    419
Name: count, dtype: int64

In [43]:
df['Key'].value_counts()

Key
7     221
9     218
2     198
0     193
D     161
A     154
G     150
4     145
11    117
C     111
B     103
5      99
E      97
G#     92
1      90
C#     88
F      82
F#     76
A#     61
6      54
10     52
8      49
3      29
D#     27
Name: count, dtype: int64

In [44]:
df['Explicit'].value_counts()

Explicit
False    1420
0.0      1204
1.0        25
True       18
Name: count, dtype: int64

In [45]:
df['Genre'].value_counts()

Genre
anime               547
classic rock        478
j-rock              448
art rock            297
progressive rock    215
indie               178
hard rock           124
alternative rock    103
indie rock           98
rock                 71
j-pop                30
Dance                16
Pop                   4
Rock                  3
celtic                2
Name: count, dtype: int64

In [46]:
# there should be 24
df['Artist ID'].nunique()

26

## **Processing values**

### **Artist ID** feature

In [47]:
artist_counts = df.groupby(['Artist ID', 'Artist name']).size().reset_index(name='Track Count')
artist_counts_sorted = artist_counts.sort_values(by='Track Count', ascending=False)
artist_counts_sorted

,Artist ID,Artist name,Track Count
15,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,297
8,1dfeR4HaWDbWqFHLkxsg1d,Queen,229
5,0k17h0D3J5VfsdmQ1iZtE9,Pink Floyd,215
24,7Ln80lUS6He07XvHI8qqHH,Arctic Monkeys,178
1,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,173
11,36QJpDe2go2KgaRleHCDTp,Led Zeppelin,163
3,0bAsR2unSRpn6BQPEnNlZm,Aimer,152
4,0blbVefuxOGltDBa00dspv,LiSA,148
18,568ZhdwyaiCyOGJRtNYhWf,Deep Purple,124
14,3w2HqkKa6upwuXEULtGvnY,FLOW,121


In [48]:
artist_id_map = {
    '6rEzedK7cKWjeQWdAYvWVG': '1dfeR4HaWDbWqFHLkxsg1d',
    '6AnrSlk5Gp1YMXgaI3mWCL': '22bE4uQ6baNwSHPVcDxLCe'
}

df['Artist ID'] = df['Artist ID'].replace(artist_id_map)

### **Mode** feature

In [49]:
def mode_convert_num(df):
    df['Mode'] = df['Mode'].replace({'Major': 1, 'Minor': 0})
    df['Mode'] = pd.to_numeric(df['Mode'], errors='coerce').astype('Int64')
    return df

In [50]:
def mode_convert_string(df):
    df['Mode'] = df['Mode'].astype(str)
    df['Mode'] = df['Mode'].replace({'1': 'Major', '0': 'Minor'})
    return df

In [51]:
df = mode_convert_string(df)

### **Key** feature

In [52]:
def replace_with_key_map(df):
    key_map = {
        'C': 0, 'C#': 1, 'D': 2, 'D#': 3,
        'E': 4, 'F': 5, 'F#': 6, 'G': 7,
        'G#': 8, 'A': 9, 'A#': 10, 'B': 11
    }

    df['Key'] = df['Key'].replace(key_map)
    df['Key'] = pd.to_numeric(df['Key'], errors='coerce').astype('Int64')
    
    return df

In [53]:
def replace_with_inv_key_map(df):
    inv_key_map = {
        '0': 'C', '1': 'C#', '2': 'D', '3': 'D#',
        '4': 'E', '5': 'F', '6': 'F#', '7': 'G',
        '8': 'G#', '9': 'A', '10': 'A#', '11': 'B'
    }
    df['Key'] = df['Key'].astype(str)
    df['Key'] = df['Key'].replace(inv_key_map)

    return df

In [54]:
df = replace_with_inv_key_map(df)

### **Explicit** feature

In [55]:
df['Explicit'] = df['Explicit'].str.lower().map(lambda x: x in ['1', '1.0', 'true', 'yes'])

### **Release date** features

In [56]:
df['Release date'] = pd.to_datetime(df['Release date'], errors='coerce')

In [57]:
# or str: .astype(str)
df['Release date'] = df['Release date'].dt.to_period('M')

### **Genre** feature

In [58]:
df['Genre'] = df['Genre'].str.lower()

In [59]:
df['Genre'] = df.apply(lambda row: 'rock' if pd.isna(row['Genre']) and row['Region'] == 'Europe' 
                       else ('j-rock' if pd.isna(row['Genre']) and row['Region'] == 'Japan' 
                             else row['Genre']), axis=1)

## **Save clean data**

In [60]:
df.dtypes

Artist ID              object
Artist name            object
Region                 object
Track ID               object
Track name             object
Release date        period[M]
Genre                  object
Popularity            float64
Explicit                 bool
Duration                int64
Acousticness          float64
Danceability          float64
Energy                float64
Instrumentalness      float64
Key                    object
Liveness              float64
Loudness              float64
Mode                   object
Speechiness           float64
Tempo                 float64
Valence               float64
dtype: object

In [61]:
df.isna().sum()

Artist ID           0
Artist name         0
Region              0
Track ID            0
Track name          0
Release date        0
Genre               0
Popularity          0
Explicit            0
Duration            0
Acousticness        0
Danceability        0
Energy              0
Instrumentalness    0
Key                 0
Liveness            0
Loudness            0
Mode                0
Speechiness         0
Tempo               0
Valence             0
dtype: int64

In [62]:
df.head()

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
0,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,70LcF31zb1H0PyJoS1Sx1r,Creep,1993-02,art rock,85.0,True,238640,0.0097,0.515,0.430,0.000133,G,0.1290,-9.935,Major,0.0372,91.844,0.104
1,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,10nyNJ6zNy2YVYLrcwLccB,No Surprises,1997-05,art rock,82.0,False,229120,0.0577,0.255,0.393,0.003610,F,0.1130,-10.654,Major,0.0278,76.426,0.118
2,0XNa1vTidXlvJ2gHSsRi4A,Franz Ferdinand,Europe,46gSk82duJtX3TTA182ruG,This fffire - New Version,2004-11,indie rock,76.0,False,218080,0.0259,0.442,0.887,0.000005,E,0.0655,-5.943,Minor,0.1000,146.358,0.637
3,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,63OQupATfueTdZMWTxW03A,Karma Police,1997-05,art rock,76.0,False,264066,0.0638,0.360,0.501,0.000093,G,0.1720,-9.129,Major,0.0258,74.807,0.324
4,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,2a1iMaoWQ5MnvLFBDv4qkf,High and Dry,1995-03,art rock,75.0,False,257480,0.0724,0.419,0.383,0.017600,E,0.0896,-11.782,Major,0.0256,87.568,0.350


In [63]:
df.to_csv("../data/cleaned_data/spotify_europeVSjapan_rock.csv", index=False)

# **Prepeared data for analysis**

In [64]:
df = pd.read_csv('../data/cleaned_data/spotify_europeVSjapan_rock.csv')
df.head()

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
0,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,70LcF31zb1H0PyJoS1Sx1r,Creep,1993-02,art rock,85.0,True,238640,0.0097,0.515,0.430,0.000133,G,0.1290,-9.935,Major,0.0372,91.844,0.104
1,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,10nyNJ6zNy2YVYLrcwLccB,No Surprises,1997-05,art rock,82.0,False,229120,0.0577,0.255,0.393,0.003610,F,0.1130,-10.654,Major,0.0278,76.426,0.118
2,0XNa1vTidXlvJ2gHSsRi4A,Franz Ferdinand,Europe,46gSk82duJtX3TTA182ruG,This fffire - New Version,2004-11,indie rock,76.0,False,218080,0.0259,0.442,0.887,0.000005,E,0.0655,-5.943,Minor,0.1000,146.358,0.637
3,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,63OQupATfueTdZMWTxW03A,Karma Police,1997-05,art rock,76.0,False,264066,0.0638,0.360,0.501,0.000093,G,0.1720,-9.129,Major,0.0258,74.807,0.324
4,4Z8W4fKeB5YxbusRsdQVPb,Radiohead,Europe,2a1iMaoWQ5MnvLFBDv4qkf,High and Dry,1995-03,art rock,75.0,False,257480,0.0724,0.419,0.383,0.017600,E,0.0896,-11.782,Major,0.0256,87.568,0.350


In [65]:
df.shape

(2667, 21)

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2667 entries, 0 to 2666
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Artist ID         2667 non-null   object 
 1   Artist name       2667 non-null   object 
 2   Region            2667 non-null   object 
 3   Track ID          2667 non-null   object 
 4   Track name        2667 non-null   object 
 5   Release date      2667 non-null   object 
 6   Genre             2667 non-null   object 
 7   Popularity        2667 non-null   float64
 8   Explicit          2667 non-null   bool   
 9   Duration          2667 non-null   int64  
 10  Acousticness      2667 non-null   float64
 11  Danceability      2667 non-null   float64
 12  Energy            2667 non-null   float64
 13  Instrumentalness  2667 non-null   float64
 14  Key               2667 non-null   object 
 15  Liveness          2667 non-null   float64
 16  Loudness          2667 non-null   float64


In [67]:
df.groupby("Artist name")["Popularity"].max()

Artist name
ASIAN KUNG-FU GENERATION    64.0
Aimer                       77.0
Arctic Monkeys              92.0
Deep Purple                 73.0
FLOW                        67.0
Franz Ferdinand             79.0
KANA-BOON                   73.0
Led Zeppelin                79.0
LiSA                        74.0
Ling tosite sigure          51.0
MY FIRST STORY              55.0
Muse                        77.0
Pink Floyd                  77.0
Queen                       82.0
Radiohead                   85.0
SID                         60.0
Sayuri                      67.0
THE ORAL CIGARETTES         69.0
The Cranberries             82.0
The Police                  86.0
The Rolling Stones          80.0
U2                          79.0
WagakkiBand                 45.0
YOASOBI                     76.0
Name: Popularity, dtype: float64

In [68]:
df_sorted = df.sort_values(by=["Artist name", "Popularity"], ascending=[True, False])
df_top_15 = df_sorted.groupby("Artist name").head(15)
df_top_15

,Artist ID,Artist name,Region,Track ID,Track name,Release date,Genre,Popularity,Explicit,Duration,Acousticness,Danceability,Energy,Instrumentalness,Key,Liveness,Loudness,Mode,Speechiness,Tempo,Valence
23,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,5ORPYXJKlpHWIdceavSGrL,遥か彼方,2012-01,j-rock,64.0,False,243346,0.000024,0.308,0.955,0.113000,E,0.1660,-3.763,Major,0.1110,174.991,0.237
34,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,0WaaPFt4Qy8sVfxKz43bCD,Re:Re:,2004-10,j-rock,59.0,False,228760,0.000318,0.537,0.864,0.123000,E,0.1080,-5.049,Major,0.0369,153.035,0.727
518,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,052C0m9kD30nZqcPWPPRqm,Haruka Kanata,2012-01,j-rock,59.0,False,243347,0.000025,0.316,0.951,0.189000,C#,0.1700,-3.757,Minor,0.1120,174.895,0.255
39,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,2sFBiCSUNc3ptKrCR8JABx,ブラッドサーキュレーター,2018-03,j-rock,57.0,False,222826,0.049300,0.372,0.975,0.000000,C,0.3310,-2.856,Major,0.1260,181.017,0.300
52,0MK8l3nURwwQIjafvXoJJt,ASIAN KUNG-FU GENERATION,Japan,6XsJTpZwPKMcG8QK8k14Z6,アフターダーク,2008-03,j-rock,57.0,False,192026,0.001930,0.387,0.950,0.000000,C#,0.0689,-3.747,Major,0.0475,191.006,0.541
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,64tJ2EAv1R6UaZqc4iOCyj,YOASOBI,Japan,2vjY3jGKElvqXoaGNEuYef,怪物,2021-12,j-pop,65.0,False,205480,0.065500,0.598,0.847,0.000099,C#,0.3070,-2.540,Major,0.2050,169.959,0.728
278,64tJ2EAv1R6UaZqc4iOCyj,YOASOBI,Japan,4BE1OloRc9xwjyqA4wFFuN,あの夢をなぞって,2020-01,j-pop,65.0,False,242666,0.217000,0.537,0.793,0.000361,G#,0.1430,-3.446,Major,0.0400,180.029,0.730
280,64tJ2EAv1R6UaZqc4iOCyj,YOASOBI,Japan,5ptl2PXkiSth54HCuGO7vN,あの夢をなぞって,2021-01,j-pop,65.0,False,240746,0.334000,0.534,0.794,0.000930,G#,0.1590,-3.738,Major,0.0402,179.976,0.711
284,64tJ2EAv1R6UaZqc4iOCyj,YOASOBI,Japan,2YbNZLoiREBYZo4HeKB8Np,ミスター,2022-02,j-pop,65.0,False,187000,0.067400,0.732,0.933,0.000037,C#,0.2020,-3.510,Minor,0.0548,119.971,0.611


In [69]:
df_top_15.to_csv("../data/cleaned_data/spotify_europeVSjapan_analysis.csv", index=False)